# Milestone 5 engineering notebook

This notebook captures the core numeric and simulation logic for the idle game milestone work.


from __future__ import annotations

from dataclasses import dataclass, field
from typing import TypedDict


class SaveResources(TypedDict):
    experience: str
    monsterSoul: str
    trainingPoints: str


class SavePlayer(TypedDict):
    level: str
    strength: str
    strengthGrowth: str


class SaveProgression(TypedDict):
    currentStage: int
    maxUnlockedStage: int
    trainingUnlocked: bool
    rebirthUnlocked: bool
    gatewayUnlocked: bool


class SaveUpgrades(TypedDict):
    weaponLevel: int
    training: dict[str, int]


class SaveAchievements(TypedDict):
    unlockedIds: list[str]


class SaveTimers(TypedDict):
    totalPlayMs: int
    trainingCycleMs: int
    rebirthCycleMs: int
    gatewayCycleMs: int
    firstTrainingMs: int | None
    firstRebirthMs: int | None
    firstGatewayMs: int | None


class SaveSchemaV1(TypedDict):
    schemaVersion: int
    meta: dict[str, object]
    resources: SaveResources
    player: SavePlayer
    progression: SaveProgression
    upgrades: SaveUpgrades
    achievements: SaveAchievements
    timers: SaveTimers


@dataclass
class GameState:
    experience: int = 0
    monsterSoul: int = 0
    trainingPoints: int = 0
    level: int = 1
    strength: int = 1
    strengthGrowth: int = 1
    currentStage: int = 1
    maxUnlockedStage: int = 1
    trainingUnlocked: bool = False
    rebirthUnlocked: bool = False
    gatewayUnlocked: bool = False
    weaponLevel: int = 0
    trainingUpgrades: dict[str, int] = field(default_factory=lambda: {
        'strengthGrowth': 0,
        'experienceModifier': 0,
        'monsterSoulModifier': 0,
    })
    achievements: list[str] = field(default_factory=list)
    totalPlayMs: int = 0
    trainingCycleMs: int = 0
    rebirthCycleMs: int = 0
    gatewayCycleMs: int = 0
    firstTrainingMs: int | None = None
    firstRebirthMs: int | None = None
    firstGatewayMs: int | None = None
    trainingResetCount: int = 0
    totalTrainingPointsEarned: int = 0
    highestStageReachedThisCycle: int = 1
    killRateWindow: list[int] = field(default_factory=list)


def clamp(value: int, minimum: int, maximum: int) -> int:
    return max(minimum, min(maximum, value))


def tier_multiplier(stage: int) -> int:
    boss_levels = {10, 100, 1000}
    return 2 ** sum(1 for threshold in boss_levels if stage >= threshold)


def monster_hit_points(stage: int) -> int:
    base = 10 * (stage * 2 + (1.08 ** stage))
    return int(base * tier_multiplier(stage))


def milestone_levels(first_milestone: int = 10, spacing: float = 1.6) -> list[int]:
    return [int(first_milestone * (spacing ** index)) for index in range(6)]


def milestone_reward(level: int) -> int:
    return max(1, int(1 + 2 * (level ** 0.5)))


def training_points_for_cycle(highest_stage: int) -> int:
    return sum(
        milestone_reward(index + 1)
        for index, milestone in enumerate(milestone_levels())
        if highest_stage >= milestone
    )


def experience_to_level(level: int, leveling_difficulty: float = 2.0) -> int:
    return max(1, int(20 * (level ** leveling_difficulty)))


def apply_level_up(state: GameState) -> None:
    while state.experience >= experience_to_level(state.level, 2.0):
        state.experience -= experience_to_level(state.level, 2.0)
        state.level += 1
        state.strength += max(1, state.strengthGrowth)


def tick(state: GameState, delta_ms: int) -> GameState:
    next_state = GameState(**state.__dict__)
    next_state.totalPlayMs += delta_ms
    next_state.trainingCycleMs += delta_ms
    next_state.rebirthCycleMs += delta_ms
    next_state.gatewayCycleMs += delta_ms
    next_state.experience += max(0, delta_ms // 1000)
    apply_level_up(next_state)
    next_state.killRateWindow = (next_state.killRateWindow + [delta_ms // 1000])[-60:]
    return next_state


# Example values for quick sanity checks
assert monster_hit_points(1) > 0
assert milestone_levels()[0] == 10
assert training_points_for_cycle(10) >= 1


from copy import deepcopy


def fresh_state() -> GameState:
    return GameState()


def snapshot(state: GameState) -> GameState:
    return deepcopy(state)


def same_state(left: GameState, right: GameState) -> bool:
    return left.__dict__ == right.__dict__


assert same_state(fresh_state(), snapshot(fresh_state()))


def deterministic_tick(state: GameState, delta_ms: int) -> GameState:
    next_state = snapshot(state)
    next_state.totalPlayMs += delta_ms
    next_state.trainingCycleMs += delta_ms
    next_state.rebirthCycleMs += delta_ms
    next_state.gatewayCycleMs += delta_ms
    next_state.experience += delta_ms // 1000
    next_state.currentStage = max(1, min(next_state.currentStage, next_state.maxUnlockedStage))
    next_state.highestStageReachedThisCycle = max(next_state.highestStageReachedThisCycle, next_state.currentStage)
    apply_level_up(next_state)
    next_state.killRateWindow = (next_state.killRateWindow + [delta_ms // 1000])[-60:]
    return next_state


baseline = fresh_state()
after_one_tick = deterministic_tick(baseline, 2000)
assert after_one_tick.experience == 2
assert after_one_tick.totalPlayMs == 2000
assert after_one_tick.killRateWindow == [2]


def can_trigger_training_reset(state: GameState) -> bool:
    return state.level >= 10 or state.currentStage >= 10


def perform_training_reset(state: GameState) -> GameState:
    next_state = snapshot(state)
    reward = training_points_for_cycle(next_state.highestStageReachedThisCycle)
    next_state.level = 1
    next_state.strength = 1
    next_state.strengthGrowth = 1 + next_state.trainingUpgrades.get('strengthGrowth', 0)
    next_state.experience = 0
    next_state.currentStage = 1
    next_state.maxUnlockedStage = 1
    next_state.monsterSoul = 0
    next_state.weaponLevel = 0
    next_state.trainingCycleMs = 0
    next_state.trainingUnlocked = True
    next_state.trainingResetCount += 1
    next_state.trainingPoints += reward
    next_state.totalTrainingPointsEarned += reward
    next_state.highestStageReachedThisCycle = 1
    next_state.killRateWindow = []
    return next_state


state_after_training = perform_training_reset(after_one_tick)
assert state_after_training.level == 1
assert state_after_training.trainingUnlocked is True
assert state_after_training.trainingPoints >= 1
assert state_after_training.trainingResetCount == 1


BOSS_THRESHOLDS = (10, 100, 1000)


def is_boss_stage(stage: int) -> bool:
    return stage in BOSS_THRESHOLDS


def can_unlock_next_stage(state: GameState, proposed_stage: int) -> bool:
    if proposed_stage <= 1:
        return True
    if proposed_stage - 1 > state.maxUnlockedStage:
        return False
    return True


def advance_stage(state: GameState, proposed_stage: int) -> GameState:
    next_state = snapshot(state)
    if not can_unlock_next_stage(next_state, proposed_stage):
        return next_state
    next_state.currentStage = proposed_stage
    next_state.maxUnlockedStage = max(next_state.maxUnlockedStage, proposed_stage)
    return next_state


assert is_boss_stage(10) is True
assert is_boss_stage(11) is False
assert can_unlock_next_stage(fresh_state(), 2) is True


ACHIEVEMENTS = [
    {'id': 'first-victory', 'title': 'First Victory', 'rewardType': 'none'},
    {'id': 'stage-10-boss', 'title': 'Stage 10 Boss', 'rewardType': 'none'},
    {'id': 'stage-100-boss', 'title': 'Stage 100 Boss', 'rewardType': 'none'},
    {'id': 'training-reset', 'title': 'Training Reset', 'rewardType': 'none'},
]


def unlock_achievements(state: GameState) -> GameState:
    next_state = snapshot(state)
    if next_state.currentStage >= 10 and 'stage-10-boss' not in next_state.achievements:
        next_state.achievements.append('stage-10-boss')
    if next_state.currentStage >= 100 and 'stage-100-boss' not in next_state.achievements:
        next_state.achievements.append('stage-100-boss')
    if next_state.trainingResetCount > 0 and 'training-reset' not in next_state.achievements:
        next_state.achievements.append('training-reset')
    return next_state


assert len(unlock_achievements(fresh_state()).achievements) == 0


def placeholder_prestige_flags(state: GameState) -> tuple[bool, bool]:
    return state.rebirthUnlocked, state.gatewayUnlocked


def update_placeholder_timers(state: GameState, delta_ms: int) -> GameState:
    next_state = snapshot(state)
    if not next_state.rebirthUnlocked:
        next_state.rebirthCycleMs += delta_ms
    if not next_state.gatewayUnlocked:
        next_state.gatewayCycleMs += delta_ms
    return next_state


assert placeholder_prestige_flags(fresh_state()) == (False, False)


# Early progression scenario
state_a = fresh_state()
state_a.currentStage = 8
state_a.maxUnlockedStage = 8
state_a = deterministic_tick(state_a, 2000)
state_a = advance_stage(state_a, 9)
assert state_a.currentStage == 9
assert state_a.maxUnlockedStage == 9

# Training loop scenario
state_b = fresh_state()
state_b.level = 10
state_b.currentStage = 10
state_b.highestStageReachedThisCycle = 10
state_b = deterministic_tick(state_b, 2000)
state_b = perform_training_reset(state_b)
assert state_b.trainingResetCount == 1
assert state_b.trainingPoints >= 1

# Boss gate scenario
state_c = fresh_state()
state_c.currentStage = 10
state_c.maxUnlockedStage = 10
state_c = unlock_achievements(state_c)
assert 'stage-10-boss' in state_c.achievements

# Placeholder flags remain locked
state_d = fresh_state()
assert placeholder_prestige_flags(state_d) == (False, False)
